# CFP_Simulation_Study

Monte Carlo simulation under controlled misspecification (Section 5.7). 5 DGPs × 2 sample sizes × 500 replications.

**Output:** `fig_simulation_qV_distribution.pdf`, `tab_simulation_extended.tex`  
**Paper:** Pele, Bolovaneanu, Ginavar, Lessmann, Härdle (2026)  
**Q** [Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle/)

In [ ]:
#!/usr/bin/env python3
"""
Monte Carlo simulation study for Section 5.7.
5 DGPs x 2 sample sizes x 500 replications.
Forecaster always assumes Normal innovations (misspecified for DGPs 2-5).
"""

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import gaussian_kde
from math import ceil
from tqdm import tqdm
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, time, shutil
warnings.filterwarnings('ignore')

t0 = time.time()

# ═════════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════════
FIG_DIR = Path('cfp_ijf_data/paper_outputs/figures')
TAB_DIR = Path('cfp_ijf_data/paper_outputs/tables')
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.01
CAL_FRAC = 0.70
N_REP = 500
T_LIST = [1000, 5000]
GARCH_OMEGA = 1e-5
GARCH_ALPHA = 0.10
GARCH_BETA = 0.85
FIT_WINDOW = 250
BASEL_WIN = 250

np.random.seed(42)

# Plot style
C_GRAY = '#666666'; C_TEAL = '#00A651'; C_RED = '#E31E24'
C_BLUE = '#0066CC'; C_PURPLE = '#7B2FBE'
plt.rcParams.update({
    'font.family': 'serif', 'axes.grid': False,
    'savefig.transparent': True, 'savefig.dpi': 300,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})

# ═════════════════════════════════════════════════════════════════════
# DGP DEFINITIONS
# ═════════════════════════════════════════════════════════════════════

DGP_LABELS = {
    1: 'Normal (correct)',
    2: 'Student-$t$(5)',
    3: 'Student-$t$(3)',
    4: 'Skewed-$t$(3)',
    5: 'Mix. Normals',
}
DGP_LABELS_PLAIN = {
    1: 'Normal (correct)',
    2: 'Student-t(5)',
    3: 'Student-t(3)',
    4: 'Skewed-t(3, -0.5)',
    5: 'Mixture 0.95*N(0,1)+0.05*N(0,5)',
}


def sample_innovations(dgp, n):
    if dgp == 1:
        return np.random.randn(n)
    elif dgp == 2:
        z = np.random.standard_t(5, size=n)
        return z / np.sqrt(5 / 3)
    elif dgp == 3:
        z = np.random.standard_t(3, size=n)
        return z / np.sqrt(3)
    elif dgp == 4:
        z = np.random.standard_t(3, size=n)
        z = z / np.sqrt(3)
        skew_factor = -0.5
        z = np.where(z < 0, z * (1 - skew_factor), z * (1 + skew_factor))
        z = (z - z.mean()) / z.std()
        return z
    elif dgp == 5:
        mask = np.random.random(n) < 0.05
        z = np.random.randn(n)
        z[mask] = np.random.randn(mask.sum()) * 5.0
        z = (z - z.mean()) / z.std()
        return z


def simulate_garch_returns(T, dgp):
    z = sample_innovations(dgp, T)
    r = np.zeros(T)
    sigma2 = np.zeros(T)
    sigma2[0] = GARCH_OMEGA / (1 - GARCH_ALPHA - GARCH_BETA)
    r[0] = np.sqrt(sigma2[0]) * z[0]
    for t in range(1, T):
        sigma2[t] = GARCH_OMEGA + GARCH_ALPHA * r[t-1]**2 + GARCH_BETA * sigma2[t-1]
        r[t] = np.sqrt(sigma2[t]) * z[t]
    return r


def fit_garch_normal_var(returns, window, alpha):
    T = len(returns)
    var_fc = np.full(T, np.nan)
    for t in range(window, T):
        r_win = returns[t - window:t]
        mu = r_win.mean()
        resid = r_win - mu
        var_unc = np.var(resid)
        omega_hat = max(var_unc * (1 - GARCH_ALPHA - GARCH_BETA), 1e-8)
        sig2 = np.zeros(window)
        sig2[0] = var_unc
        for i in range(1, window):
            sig2[i] = omega_hat + GARCH_ALPHA * resid[i-1]**2 + GARCH_BETA * sig2[i-1]
        sigma2_next = omega_hat + GARCH_ALPHA * resid[-1]**2 + GARCH_BETA * sig2[-1]
        sigma_next = np.sqrt(max(sigma2_next, 1e-10))
        var_fc[t] = mu + sigma_next * stats.norm.ppf(alpha)
    return var_fc


def conformal_correct(ret_cal, var_cal, ret_test, var_test, alpha):
    scores = var_cal - ret_cal
    n = len(scores)
    k = min(ceil((n + 1) * (1 - alpha)), n)
    qV = np.sort(scores)[k - 1]
    corrected = var_test - qV
    return corrected, qV


def basel_tl(n_viol, n_test):
    s = n_viol * (BASEL_WIN / n_test) if n_test > 0 else 0
    if s <= 4: return 'Green'
    elif s <= 9: return 'Yellow'
    return 'Red'


# ═════════════════════════════════════════════════════════════════════
# MAIN SIMULATION LOOP
# ═════════════════════════════════════════════════════════════════════
print('Monte Carlo Simulation Study (Section 5.7)')
print(f'  DGPs: 5, T: {T_LIST}, Replications: {N_REP}')
print(f'  GARCH(1,1): omega={GARCH_OMEGA}, alpha={GARCH_ALPHA}, beta={GARCH_BETA}')
print(f'  Forecaster: always GARCH-Normal (misspecified for DGPs 2-5)\n')

results = []
qV_store = {(dgp, T): [] for T in T_LIST for dgp in range(1, 6)}

for T in T_LIST:
    for dgp in range(1, 6):
        key = (dgp, T)
        desc = f'DGP {dgp} ({DGP_LABELS_PLAIN[dgp][:20]:20s}), T={T}'
        for rep in tqdm(range(N_REP), desc=desc, leave=True):
            returns = simulate_garch_returns(T, dgp)
            var_fc = fit_garch_normal_var(returns, FIT_WINDOW, ALPHA)

            valid = ~np.isnan(var_fc)
            r_v = returns[valid]
            v_v = var_fc[valid]
            n_v = len(r_v)
            if n_v < 100:
                continue

            cal_end = int(n_v * CAL_FRAC)
            r_cal, v_cal = r_v[:cal_end], v_v[:cal_end]
            r_test, v_test = r_v[cal_end:], v_v[cal_end:]
            n_test = len(r_test)
            if n_test < 50:
                continue

            viol_raw = (r_test < v_test).astype(int)
            pi_raw = viol_raw.mean()
            tl_raw = basel_tl(int(viol_raw.sum()), n_test)

            v_corr, qV = conformal_correct(r_cal, v_cal, r_test, v_test, ALPHA)
            viol_corr = (r_test < v_corr).astype(int)
            pi_corr = viol_corr.mean()
            tl_corr = basel_tl(int(viol_corr.sum()), n_test)

            results.append({
                'dgp': dgp, 'T': T, 'rep': rep,
                'qV': qV, 'pi_raw': pi_raw, 'pi_corr': pi_corr,
                'tl_raw': tl_raw, 'tl_corr': tl_corr,
                'n_cal': cal_end, 'n_test': n_test,
            })
            qV_store[key].append(qV)

df = pd.DataFrame(results)
print(f'\nTotal results: {len(df)} rows')

# Save raw qV arrays for KDE figure
import pickle
with open(TAB_DIR / 'simulation_qV_raw.pkl', 'wb') as _f:
    pickle.dump(qV_store, _f)
print(f'Saved raw qV arrays to {TAB_DIR}/simulation_qV_raw.pkl')

# ═════════════════════════════════════════════════════════════════════
# OUTPUT TABLE
# ═════════════════════════════════════════════════════════════════════
print('\n' + '=' * 80)
print('SIMULATION RESULTS')
print('=' * 80)

tab_rows = []
for T in T_LIST:
    for dgp in range(1, 6):
        sub = df[(df['dgp'] == dgp) & (df['T'] == T)]
        if len(sub) == 0:
            continue
        tab_rows.append({
            'DGP': dgp, 'Label': DGP_LABELS_PLAIN[dgp], 'T': T,
            'Mean_qV': sub['qV'].mean(), 'Std_qV': sub['qV'].std(),
            'Raw_pihat': sub['pi_raw'].mean(), 'Corr_pihat': sub['pi_corr'].mean(),
            'Raw_Green_pct': (sub['tl_raw'] == 'Green').mean() * 100,
            'Corr_Green_pct': (sub['tl_corr'] == 'Green').mean() * 100,
        })

df_tab = pd.DataFrame(tab_rows)
df_tab.to_csv(TAB_DIR / 'simulation_results.csv', index=False)
print(df_tab.round(4).to_string(index=False))

# LaTeX table
latex_lines = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Monte Carlo simulation: conformal correction under controlled misspecification (500 replications, $\alpha = 0.01$, $f_c = 0.70$). The forecaster always assumes Normal innovations.}',
    r'\label{tab:simulation_extended}',
    r'\footnotesize',
    r'\begin{tabular}{@{}clc rr rr rr@{}}',
    r'\toprule',
    r'& & & \multicolumn{2}{c}{$\hat{q}_V$} & \multicolumn{2}{c}{$\hat\pi$} & \multicolumn{2}{c}{Green \%} \\',
    r'\cmidrule(lr){4-5}\cmidrule(lr){6-7}\cmidrule(l){8-9}',
    r'DGP & Innovations & $T$ & Mean & Std & Raw & Corr. & Raw & Corr. \\',
    r'\midrule',
]

prev_dgp = None
for _, row in df_tab.iterrows():
    dgp = int(row['DGP'])
    if prev_dgp is not None and dgp != prev_dgp:
        latex_lines.append(r'\addlinespace')
    prev_dgp = dgp
    label = DGP_LABELS[dgp]
    T = int(row['T'])
    latex_lines.append(
        f"{dgp} & {label} & {T:,} & {row['Mean_qV']:.4f} & {row['Std_qV']:.4f} "
        f"& {row['Raw_pihat']:.3f} & {row['Corr_pihat']:.3f} "
        f"& {row['Raw_Green_pct']:.0f} & {row['Corr_Green_pct']:.0f} \\\\"
    )

latex_lines += [
    r'\bottomrule',
    r'\end{tabular}',
    r'\begin{minipage}{\linewidth}\footnotesize',
    r'\smallskip',
    r'DGP~1: Normal (correctly specified). DGP~2: Student-$t$(5). DGP~3: Student-$t$(3). '
    r'DGP~4: Skewed-$t$(3, $-0.5$). DGP~5: Mixture $0.95 \mathcal{N}(0,1) + 0.05\mathcal{N}(0,25)$. '
    r'GARCH(1,1) parameters: $\omega = 10^{-5}$, $\alpha = 0.10$, $\beta = 0.85$.',
    r'\end{minipage}',
    r'\end{table}',
]

tex_content = '\n'.join(latex_lines)
(TAB_DIR / 'tab_simulation_extended.tex').write_text(tex_content)
print(f'\nSaved: {TAB_DIR}/tab_simulation_extended.tex')

# Copy to root for \input{}
Path('tab_simulation_extended.tex').write_text(tex_content)
print('Saved: tab_simulation_extended.tex (root copy)')

# ═════════════════════════════════════════════════════════════════════
# OUTPUT FIGURE
# ═════════════════════════════════════════════════════════════════════
dgp_colors = {1: C_GRAY, 2: C_BLUE, 3: C_TEAL, 4: C_PURPLE, 5: C_RED}

fig, axes = plt.subplots(5, 1, figsize=(5, 14), sharex=True)

for i, dgp in enumerate(range(1, 6)):
    ax = axes[i]
    for T, ls in zip(T_LIST, ['-', '--']):
        vals = np.array(qV_store.get((dgp, T), []))
        if len(vals) < 10 or vals.std() == 0:
            continue
        kde = gaussian_kde(vals)
        x_grid = np.linspace(vals.min() - 0.005, vals.max() + 0.005, 200)
        ax.fill_between(x_grid, kde(x_grid), alpha=0.25, color=dgp_colors[dgp])
        ax.plot(x_grid, kde(x_grid), color=dgp_colors[dgp],
                linestyle=ls, linewidth=1.2, label=f'$T={T:,}$')

    ax.axvline(0, color='black', linewidth=0.7, linestyle=':')
    ax.set_title(f'DGP {dgp}: {DGP_LABELS[dgp]}', fontsize=8)
    if i == 4:
        ax.set_xlabel('$\\hat{q}_V$', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=7, frameon=False, loc='upper right')
    ax.tick_params(labelsize=7)

fig.suptitle('Distribution of $\\hat{q}_V$ Across 500 MC Replications per DGP',
             fontsize=11, y=1.03)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_simulation_qV_distribution.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / 'fig_simulation_qV_distribution.png', bbox_inches='tight')
plt.close(fig)
print(f'Saved: {FIG_DIR}/fig_simulation_qV_distribution.pdf/png')

# Copy to figures/ for LaTeX
shutil.copy(FIG_DIR / 'fig_simulation_qV_distribution.pdf',
            Path('figures') / 'fig_simulation_qV_distribution.pdf')
print('Copied to figures/fig_simulation_qV_distribution.pdf')

# ═════════════════════════════════════════════════════════════════════
# SUMMARY
# ═════════════════════════════════════════════════════════════════════
elapsed = time.time() - t0
print(f'\nElapsed: {elapsed:.1f}s')
print('\nKey findings:')
for dgp in range(1, 6):
    s1 = df[(df['dgp'] == dgp) & (df['T'] == 1000)]
    s5 = df[(df['dgp'] == dgp) & (df['T'] == 5000)]
    if len(s1) > 0 and len(s5) > 0:
        print(f'  DGP {dgp} ({DGP_LABELS_PLAIN[dgp][:25]:25s}): '
              f'qV = {s1["qV"].mean():.4f} (T=1k) -> {s5["qV"].mean():.4f} (T=5k), '
              f'Green = {(s1["tl_corr"]=="Green").mean()*100:.0f}% -> '
              f'{(s5["tl_corr"]=="Green").mean()*100:.0f}%')
print('\nDone.')
